# Dump per-seed test predictions for the 4 v02 variants

Inference-only — no retraining. Loads each checkpoint from
`paper_revision_v01/models/...`, runs test inference, and writes
`predictions_test.csv` to `paper_revision_v03/eval_results_variants/{variant}/seed_{k}/`.

Combined with the v03 results already on disk
(`vit_imagenet_gated`, `vit_dino_cls`), this gives **per-seed predictions for
all 6 variants in one place**, which is what's needed to regenerate Fig 7 with
mean curves + ±1 SD shaded bands.

## Variants dumped
| Variant | Checkpoint path |
|---|---|
| `proposed`         | `paper_revision_v01/models/baseline_model_seed{k}.pth` |
| `resnet50_gated`   | `paper_revision_v01/models/resnet50_gated/baseline_model_seed{k}.pth` |
| `vit_dino_mean`    | `paper_revision_v01/models/vit_dino_mean/baseline_model_seed{k}.pth` |
| `vit_dino_nogate`  | `paper_revision_v01/models/vit_dino_nogate/baseline_model_seed{k}.pth` |

## Run order
1. **Cell 1** — Drive mount + imports + paths.
2. **Cell 2** — dataset + transforms.
3. **Cell 3** — `FociClassifierVariant` model class (same as v03's, supports gated / attention-no-gate / mean / cls).
4. **Cell 4** — inference loop. Idempotent: skips any (variant, seed) whose `predictions_test.csv` already exists.


In [ ]:
# =========================
# Cell 1 — setup + paths
# =========================
import os, json, torch, numpy as np, pandas as pd, timm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

# Read-only inputs ---------------------------------------------------------
VAL_CSV       = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/val_dataset.csv"
TEST_CSV      = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/test_dataset.csv"
BACKBONE_CKPT = "/content/drive/MyDrive/FYP/natural foci ground truths/backbone models/CELL_ONLY_foci_dino_backbone_DINOv03_v01.pth"

# v02 model checkpoints (already on Drive) --------------------------------
V01_MODELS_DIR = "/content/drive/MyDrive/FYP/paper_revision_v01/models"

# Write target (paper_revision_v03, alongside the v03 variants) -----------
EVAL_DIR = "/content/drive/MyDrive/FYP/paper_revision_v03/eval_results_variants"
os.makedirs(EVAL_DIR, exist_ok=True)

SIZE_MODE = "384"
CFG = {
    "batch_size":    32,
    "image_size":    int(SIZE_MODE),
    "backbone_name": f"vit_tiny_patch16_{SIZE_MODE}",
    "device":        torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}
DEVICE = CFG["device"]

print(f"Device       : {DEVICE}")
print(f"Test CSV     : {TEST_CSV}")
print(f"Writes to    : {EVAL_DIR}")
print(f"Checkpoints  : {V01_MODELS_DIR}")


In [ ]:
# =========================
# Cell 2 — eval dataset + transforms
# =========================
NORM_MEAN = [0.2251, 0.2251, 0.2251]
NORM_STD  = [0.2375, 0.2375, 0.2375]

eval_transform = transforms.Compose([
    transforms.Resize((CFG["image_size"], CFG["image_size"]),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])


class FociEvalDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(str(row["image_path"])).convert("RGB")
            return (eval_transform(img),
                    torch.tensor(float(row["label"])).float(),
                    str(row["image_path"]))
        except Exception:
            return None


def safe_collate_eval(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    xs, ys, paths = zip(*batch)
    return torch.stack(xs), torch.stack(ys), list(paths)


In [ ]:
# =========================
# Cell 3 — model class (same as v03's)
# =========================
class FociClassifierVariant(nn.Module):
    def __init__(self, backbone_name, backbone_ckpt, img_size,
                 pooling_mode, use_imagenet=False):
        super().__init__()
        self.pooling_mode  = pooling_mode
        self.backbone_name = backbone_name
        self.is_vit        = backbone_name.startswith("vit_")

        if self.is_vit:
            self.backbone = timm.create_model(
                backbone_name, pretrained=use_imagenet,
                num_classes=0, global_pool="", img_size=img_size,
            )
            if backbone_ckpt is not None and os.path.exists(backbone_ckpt):
                sd = torch.load(backbone_ckpt, map_location="cpu")
                self.backbone.load_state_dict(
                    {k.replace("backbone.", "").replace("vit.", ""): v
                     for k, v in sd.items()},
                    strict=False,
                )
        else:
            self.backbone = timm.create_model(
                backbone_name, pretrained=use_imagenet,
                num_classes=0, global_pool="",
            )

        for p in self.backbone.parameters():
            p.requires_grad = False

        with torch.no_grad():
            feats = self.backbone.forward_features(torch.zeros(1, 3, img_size, img_size))
            if isinstance(feats, dict): feats = feats["x"]
            dim = feats.shape[-1] if self.is_vit else feats.shape[1]

        hidden_dim = 128
        self.head_norm = nn.LayerNorm(dim)
        if pooling_mode == "gated_attention":
            self.attention_V = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh())
            self.attention_U = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Sigmoid())
            self.attention_w = nn.Linear(hidden_dim, 1)
        elif pooling_mode == "attention_nogate":
            self.attention_V = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh())
            self.attention_w = nn.Linear(hidden_dim, 1)

        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1),
        )

    def _features(self, x):
        feats = self.backbone.forward_features(x)
        if isinstance(feats, dict): feats = feats["x"]
        if self.is_vit:
            return {"cls": feats[:, 0, :], "patches": feats[:, 1:, :]}
        return {"map": feats}

    def forward(self, x):
        f = self._features(x)
        if self.pooling_mode == "cls":
            if self.is_vit:
                pooled = self.head_norm(f["cls"])
            else:
                pooled = self.head_norm(f["map"].mean(dim=[2, 3]))
            return self.classifier(pooled).squeeze(-1)

        if self.is_vit:
            patches = f["patches"]
        else:
            m = f["map"]; B, C, H, W = m.shape
            patches = m.reshape(B, C, H*W).permute(0, 2, 1)
        normed = self.head_norm(patches)

        if self.pooling_mode == "gated_attention":
            A = torch.softmax(self.attention_w(
                self.attention_V(normed) * self.attention_U(normed)
            ).squeeze(-1), dim=1)
            pooled = torch.sum(patches * A.unsqueeze(-1), dim=1)
        elif self.pooling_mode == "attention_nogate":
            A = torch.softmax(
                self.attention_w(self.attention_V(normed)).squeeze(-1), dim=1)
            pooled = torch.sum(patches * A.unsqueeze(-1), dim=1)
        else:  # mean
            pooled = normed.mean(dim=1)

        return self.classifier(pooled).squeeze(-1)


In [ ]:
# =========================
# Cell 4 — dump per-seed test predictions for the 4 v02 variants
# =========================
N_SEEDS = 5

# Each entry: (variant_name, ckpt_template, backbone, pooling, use_imagenet)
#   ckpt_template uses {seed} placeholder.
VARIANTS = [
    {
        "name":          "proposed",
        "backbone_name": CFG["backbone_name"],
        "use_imagenet":  False,
        "pooling":       "gated_attention",
        "ckpt_template": os.path.join(V01_MODELS_DIR, "baseline_model_seed{seed}.pth"),
    },
    {
        "name":          "resnet50_gated",
        "backbone_name": "resnet50",
        "use_imagenet":  False,
        "pooling":       "gated_attention",
        "ckpt_template": os.path.join(V01_MODELS_DIR, "resnet50_gated",
                                      "baseline_model_seed{seed}.pth"),
    },
    {
        "name":          "vit_dino_mean",
        "backbone_name": CFG["backbone_name"],
        "use_imagenet":  False,
        "pooling":       "mean",
        "ckpt_template": os.path.join(V01_MODELS_DIR, "vit_dino_mean",
                                      "baseline_model_seed{seed}.pth"),
    },
    {
        "name":          "vit_dino_nogate",
        "backbone_name": CFG["backbone_name"],
        "use_imagenet":  False,
        "pooling":       "attention_nogate",
        "ckpt_template": os.path.join(V01_MODELS_DIR, "vit_dino_nogate",
                                      "baseline_model_seed{seed}.pth"),
    },
]


def infer_split(model, csv_path):
    loader = DataLoader(
        FociEvalDataset(csv_path),
        batch_size=CFG["batch_size"], shuffle=False,
        collate_fn=safe_collate_eval, num_workers=0,
    )
    all_probs, all_labels, all_paths = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            if batch is None: continue
            x, y, paths = batch
            x = x.to(DEVICE)
            probs = torch.sigmoid(model(x))
            all_probs.extend(probs.cpu().numpy().tolist())
            all_labels.extend(y.cpu().numpy().tolist())
            all_paths.extend(paths)
    return (np.array(all_labels).astype(int),
            np.array(all_probs).astype(float),
            all_paths)


print("Dumping per-seed val + test predictions for v02 variants...")
for v in VARIANTS:
    name = v["name"]
    for seed in range(N_SEEDS):
        out_dir   = os.path.join(EVAL_DIR, name, f"seed_{seed}")
        test_csv_out = os.path.join(out_dir, "predictions_test.csv")
        val_csv_out  = os.path.join(out_dir, "predictions_val.csv")
        if os.path.exists(test_csv_out) and os.path.exists(val_csv_out):
            print(f"[skip] {name} seed {seed} — both CSVs exist")
            continue

        ckpt = v["ckpt_template"].format(seed=seed)
        if not os.path.exists(ckpt):
            print(f"[MISSING CKPT] {name} seed {seed}: {ckpt}")
            continue

        print(f"\n--- {name} seed {seed} ---")
        backbone_ckpt = BACKBONE_CKPT if (v["pooling"] in ("gated_attention", "mean",
                                                            "attention_nogate", "cls")
                                          and v["backbone_name"].startswith("vit_")) \
                                       else None
        model = FociClassifierVariant(
            backbone_name=v["backbone_name"],
            backbone_ckpt=backbone_ckpt,
            img_size=CFG["image_size"],
            pooling_mode=v["pooling"],
            use_imagenet=v["use_imagenet"],
        ).to(DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE), strict=False)
        model.eval()

        os.makedirs(out_dir, exist_ok=True)

        # val predictions (for val-tuned threshold)
        if not os.path.exists(val_csv_out):
            y_v, p_v, paths_v = infer_split(model, VAL_CSV)
            pd.DataFrame({"image_path": paths_v, "label": y_v, "prob": p_v}) \
                .to_csv(val_csv_out, index=False)
            print(f"  wrote {val_csv_out}  ({len(y_v)} rows)")

        # test predictions (for headline metrics + Fig 7 curves + bands)
        if not os.path.exists(test_csv_out):
            y_t, p_t, paths_t = infer_split(model, TEST_CSV)
            pd.DataFrame({"image_path": paths_t, "label": y_t, "prob": p_t}) \
                .to_csv(test_csv_out, index=False)
            print(f"  wrote {test_csv_out}  ({len(y_t)} rows)")

        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print("\nDone. Per-seed CSVs (val + test) are at:")
for v in VARIANTS:
    print(f"  {os.path.join(EVAL_DIR, v['name'])}/seed_*/predictions_{{val,test}}.csv")
print("\nThese, together with the existing v03 entries for")
print("  vit_imagenet_gated/  and  vit_dino_cls/")
print("cover all 6 variants needed to regenerate Fig 7 with mean curves + SD bands.")
